In [1]:
import sys
sys.path.insert(0, r"D:\BDA\nb" if sys.platform == "win32" else "/workspace/nb")
from bda_common import *

from pyspark.sql import functions as F
from pyspark.ml.feature import CountVectorizer, CountVectorizerModel, IDF, Normalizer
from pyspark.ml.linalg import SparseVector, VectorUDT

info_mesin()
spark = spark_session("04-kmer-features")
print()
print("  Sumber :", jalur("stage/sequences"))
print("  Tujuan :", jalur("features"))


CPU logis      : 24
RAM total      : 50.5 GB   bebas 40.3 GB
Disk D: bebas  : 201.4 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20260922T024815Z


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 02:48:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 02:48:17 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark 3.5.9 | master yarn | driver 8g | paralelisme 8
Spark UI: http://jupyter:4040

  Sumber : hdfs://namenode:8020/bda/stage/sequences
  Tujuan : hdfs://namenode:8020/bda/features


In [2]:
with Tahap("baca zona silver", "FEATURE"):
    silver = spark.read.parquet(jalur("stage/sequences"))
    n_total = silver.count()
    print(f"  Baris silver seluruhnya : {n_total:,}")

    unik = silver.filter(F.col("wakil_unik"))
    n_unik = unik.count()
    print(f"  Sekuens unik            : {n_unik:,}  ({100*n_unik/max(n_total,1):.1f}%)")
    print(f"  Salinan identik         : {n_total - n_unik:,}  "
          f"-> disambungkan lagi di notebook 06 lewat hash_seq")

    berlabel = unik.filter(F.col("segmen").isNotNull())
    tanpa_label = unik.filter(F.col("segmen").isNull())
    n_lab, n_tak = berlabel.count(), tanpa_label.count()
    print(f"\n  Segmen diketahui        : {n_lab:,}")
    print(f"  Segmen tidak diketahui  : {n_tak:,}  (sasaran imputasi notebook 05)")

    print("\n  Distribusi panjang sekuens:")
    silver.select(
        F.min("panjang").alias("min"),
        F.expr("percentile_approx(panjang, 0.25)").alias("p25"),
        F.expr("percentile_approx(panjang, 0.50)").alias("median"),
        F.expr("percentile_approx(panjang, 0.75)").alias("p75"),
        F.max("panjang").alias("max"),
        F.round(F.avg("panjang"), 1).alias("rata2")).show()



--------------------------------------------------------------------
[>] FEATURE | baca zona silver


  Baris silver seluruhnya : 1,586,912
  Sekuens unik            : 893,599  (56.3%)
  Salinan identik         : 693,313  -> disambungkan lagi di notebook 06 lewat hash_seq

  Segmen diketahui        : 892,344
  Segmen tidak diketahui  : 1,255  (sasaran imputasi notebook 05)

  Distribusi panjang sekuens:


+---+----+------+----+----+------+
|min| p25|median| p75| max| rata2|
+---+----+------+----+----+------+
|500|1025|  1541|2208|2600|1619.8|
+---+----+------+----+----+------+

[<] OK | 8.9 detik | RAM bebas 37.6 GB


In [3]:
def ekstrak_kmer(df, k):
    gen = f"transform(sequence(0, length(seq) - {k}), i -> substring(seq, i + 1, {k}))"
    ek = (f"CASE WHEN n_acgt = panjang THEN {gen} "
          f"ELSE filter({gen}, x -> length(translate(x, 'ACGT', '')) = 0) END")
    return df.withColumn("kmers", F.expr(ek)).select("accession", "kmers")

with Tahap("demonstrasi jendela geser", "FEATURE"):
    contoh = silver.select("accession", "seq", "n_acgt", "panjang").limit(1)
    baris = contoh.collect()[0]
    print(f"  Aksesi  : {baris['accession']}")
    print(f"  Sekuens : {baris['seq'][:60]} ... (panjang {len(baris['seq'])})")
    k_demo = CFG["k_utama"]
    print()
    print(f"  k = {k_demo}, {len(baris['seq']) - k_demo + 1:,} k-mer dihasilkan:")
    for i in range(5):
        print(f"    posisi {i+1:>2} -> {baris['seq'][i:i+k_demo]}")
    print("    ...")

    uji = ekstrak_kmer(contoh, k_demo).collect()[0]
    print()
    print(f"  Diverifikasi lewat Spark: {len(uji['kmers']):,} k-mer")
    print(f"  Lima pertama            : {uji['kmers'][:5]}")

    bersih = silver.filter(F.col("n_acgt") == F.col("panjang")).count()
    print()
    print(f"  Sekuens tanpa huruf ambigu sama sekali: {bersih:,} "
          f"({100*bersih/max(n_total,1):.1f}%)")
    print("  -> sebanyak itu melewati penyaringan per-k-mer sepenuhnya")



--------------------------------------------------------------------
[>] FEATURE | demonstrasi jendela geser
  Aksesi  : MT090350
  Sekuens : AATATGGAGAGAATAAAAGAACTAAGAGATTTGATGTCGCAGTCTCGCACTCGCGAGATA ... (panjang 2292)

  k = 8, 2,285 k-mer dihasilkan:
    posisi  1 -> AATATGGA
    posisi  2 -> ATATGGAG
    posisi  3 -> TATGGAGA
    posisi  4 -> ATGGAGAG
    posisi  5 -> TGGAGAGA
    ...



  Diverifikasi lewat Spark: 2,285 k-mer
  Lima pertama            : ['AATATGGA', 'ATATGGAG', 'TATGGAGA', 'ATGGAGAG', 'TGGAGAGA']

  Sekuens tanpa huruf ambigu sama sekali: 1,503,183 (94.7%)
  -> sebanyak itu melewati penyaringan per-k-mer sepenuhnya
[<] OK | 1.6 detik | RAM bebas 37.2 GB


In [4]:
from pyspark import StorageLevel

def sudah_ada(k):
    try:
        return spark.read.parquet(jalur(f"features/kmer_k{k}")).limit(1).count() > 0
    except Exception:
        return False

def bangun_fitur(k, fraksi_sampel=0.1, simpan=True):
    dim_penuh = 4 ** k
    pakai_hashing = dim_penuh > CFG["vocab_max"]
    vocab_size = min(dim_penuh, CFG["vocab_max"])
    tujuan = jalur(f"features/kmer_k{k}")

    print()
    print(f"  k={k}: dimensi penuh 4^{k} = {dim_penuh:,}"
          f" -> vocabSize {vocab_size:,}"
          f"{'  (dibatasi, k-mer paling jarang tersaring minDF)' if pakai_hashing else ''}")

    if simpan and sudah_ada(k):
        n = spark.read.parquet(tujuan).count()
        print(f"    sudah ada -- dilewati ({n:,} baris)")
        return {"k": k, "dim_penuh": dim_penuh, "vocab": None,
                "baris": n, "tujuan": tujuan, "dilewati": True}

    sumber = berlabel.select("accession", "seq", "n_acgt", "panjang")

    sampel = ekstrak_kmer(sumber.sample(False, fraksi_sampel, seed=CFG["seed"]), k)
    cv = CountVectorizer(inputCol="kmers", outputCol="tf",
                         vocabSize=vocab_size, minDF=2.0)
    cv_model = cv.fit(sampel)
    print(f"    kosakata dipelajari : {len(cv_model.vocabulary):,} k-mer unik")

    tok = ekstrak_kmer(sumber, k)
    tf = cv_model.transform(tok).drop("kmers")
    tf = tf.persist(StorageLevel.MEMORY_AND_DISK)
    n_tf = tf.count()
    print(f"    vektor TF disimpan  : {n_tf:,} baris")

    idf = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=2)
    idf_model = idf.fit(tf)
    fitur = idf_model.transform(tf).drop("tf")

    norm = Normalizer(inputCol="tfidf", outputCol="features", p=2.0)
    fitur = norm.transform(fitur).drop("tfidf")

    label = berlabel.select("accession", "hash_seq", "n_duplikat_seq",
                            "segmen", "subtipe", "inang",
                            "negara", "wilayah", "tahun_koleksi", "panjang",
                            "isolat", "segmen_part")
    hasil = fitur.join(label, on="accession", how="inner")

    if simpan:
        (hasil.write.mode("overwrite")
              .partitionBy("segmen_part")
              .parquet(tujuan))
        cv_model.write().overwrite().save(jalur(f"features/vocab_k{k}"))
        n = spark.read.parquet(tujuan).count()
    else:
        n = hasil.count()

    tf.unpersist()

    return {"k": k, "dim_penuh": dim_penuh, "vocab": len(cv_model.vocabulary),
            "baris": n, "tujuan": tujuan, "dilewati": False}


In [5]:
K_DIJALANKAN = CFG["k_ablasi"]

ringkasan = []
with Tahap(f"bangun feature store untuk k={K_DIJALANKAN}", "FEATURE"):
    for k in K_DIJALANKAN:
        t0 = time.time()
        r = bangun_fitur(k)
        r["detik"] = round(time.time() - t0, 1)
        ringkasan.append(r)
        print(f"    -> {r['baris']:,} baris dalam {r['detik']:,.1f} detik")

import pandas as pd
df_ablasi = pd.DataFrame(ringkasan)
display(df_ablasi)
(BASE / "features" / "ringkasan_ablasi.json").write_text(
    json.dumps(ringkasan, indent=2), encoding="utf-8")



--------------------------------------------------------------------
[>] FEATURE | bangun feature store untuk k=[4, 6, 8, 10]

  k=4: dimensi penuh 4^4 = 256 -> vocabSize 256
    sudah ada -- dilewati (892,344 baris)
    -> 892,344 baris dalam 1.0 detik

  k=6: dimensi penuh 4^6 = 4,096 -> vocabSize 4,096


    sudah ada -- dilewati (892,344 baris)
    -> 892,344 baris dalam 2.6 detik

  k=8: dimensi penuh 4^8 = 65,536 -> vocabSize 65,536


    sudah ada -- dilewati (892,344 baris)
    -> 892,344 baris dalam 2.1 detik

  k=10: dimensi penuh 4^10 = 1,048,576 -> vocabSize 262,144  (dibatasi, k-mer paling jarang tersaring minDF)


[Stage 50:====================================================> (172 + 6) / 178]

    sudah ada -- dilewati (892,344 baris)
    -> 892,344 baris dalam 2.2 detik
[<] OK | 7.8 detik | RAM bebas 36.2 GB


,k,dim_penuh,vocab,baris,tujuan,dilewati,detik
0,4,256,None,892344,hdfs://namenode:8020/bda/features/kmer_k4,True,1.0
1,6,4096,None,892344,hdfs://namenode:8020/bda/features/kmer_k6,True,2.6
2,8,65536,None,892344,hdfs://namenode:8020/bda/features/kmer_k8,True,2.1
3,10,1048576,None,892344,hdfs://namenode:8020/bda/features/kmer_k10,True,2.2


735

In [6]:
with Tahap("ukur kejarangan vektor fitur", "FEATURE"):
    baris_ukur = []
    for k in K_DIJALANKAN:

        SKEMA_MENTAH = ("accession string, features "
                        "struct<type:tinyint,size:int,"
                        "indices:array<int>,values:array<double>>")
        f = spark.read.schema(SKEMA_MENTAH).parquet(jalur(f"features/kmer_k{k}"))

        nnz = (F.when(F.col("features.type") == 0, F.size("features.indices"))
                .otherwise(F.size(F.filter(F.col("features.values"),
                                           lambda x: x != 0.0))))
        s = (f.select(nnz.alias("nnz"))
              .agg(F.avg("nnz").alias("rata2"),
                   F.min("nnz").alias("min"),
                   F.max("nnz").alias("max"),
                   F.sum("nnz").alias("total")).collect()[0])
        dim = 4 ** k
        baris_ukur.append({
            "k": k, "dimensi": dim,
            "nnz_rata2": round(s["rata2"], 1),
            "kepadatan_pct": round(100 * s["rata2"] / dim, 4),
            "total_nnz": int(s["total"]),
            "ukuran_jarang_GB": round(s["total"] * 8 / 1e9, 2),
            "ukuran_padat_TB": round(f.count() * dim * 4 / 1e12, 1),
        })

    df_sparsity = pd.DataFrame(baris_ukur)
    display(df_sparsity)
    print("\n  Kolom ukuran_padat_TB memperlihatkan berapa besar matriks ini")
    print("  seandainya disimpan padat -- alasan mengapa representasi jarang")
    print("  bukan optimasi tambahan, melainkan syarat agar pekerjaan ini mungkin.")



--------------------------------------------------------------------
[>] FEATURE | ukur kejarangan vektor fitur


,k,dimensi,nnz_rata2,kepadatan_pct,total_nnz,ukuran_jarang_GB,ukuran_padat_TB
0,4,256,244.4,95.4673,218085674,1.74,0.0
1,6,4096,1223.2,29.8627,1091494463,8.73,0.0
2,8,65536,1629.8,2.4869,1454356764,11.63,0.2
3,10,1048576,1649.3,0.1573,1471716261,11.77,3.7



  Kolom ukuran_padat_TB memperlihatkan berapa besar matriks ini
  seandainya disimpan padat -- alasan mengapa representasi jarang
  bukan optimasi tambahan, melainkan syarat agar pekerjaan ini mungkin.
[<] OK | 56.2 detik | RAM bebas 30.6 GB


In [7]:
with Tahap("fitur untuk sekuens tanpa label", "FEATURE"):
    if n_tak == 0:
        print("  Tidak ada sekuens tanpa label segmen -- tidak ada yang perlu diimputasi.")
    else:
        k = CFG["k_utama"]
        cv_model = CountVectorizerModel.load(jalur(f"features/vocab_k{k}"))

        tok = ekstrak_kmer(
            tanpa_label.select("accession", "seq", "n_acgt", "panjang"), k)
        tf = cv_model.transform(tok).drop("kmers")
        idf = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=1).fit(tf)
        v = Normalizer(inputCol="tfidf", outputCol="features", p=2.0) \
            .transform(idf.transform(tf)).select("accession", "features")
        hasil = v.join(tanpa_label.select("accession", "defline", "subtipe",
                                          "panjang"), on="accession")
        hasil.write.mode("overwrite").parquet(jalur("features/tanpa_label_segmen"))
        print(f"  {hasil.count():,} sekuens siap diimputasi di notebook 05")



--------------------------------------------------------------------
[>] FEATURE | fitur untuk sekuens tanpa label


26/09/22 02:50:00 WARN DAGScheduler: Broadcasting large task binary with size 1763.8 KiB
                                                                                

  1,255 sekuens siap diimputasi di notebook 05
[<] OK | 35.4 detik | RAM bebas 30.6 GB


In [8]:
display(ringkas_zona())
display(jejak_df())
stop_spark()
print("\nSelesai. Lanjut ke 05_ml_segmen_subtipe.ipynb")


,zona,isi,ukuran
0,lake/fasta,152,162.2 MB
1,lake/meta_csv,49,367.6 MB
2,stage,1,1.3 KB
3,features,1,735.0 B
4,models,4,2.2 KB
5,graph,0,0.0 B
6,mart,0,0.0 B
7,output,3,237.4 KB


,run_id,lapisan,tahap,status,detik,ram_delta_gb,ram_bebas_gb,waktu
0,run_20260922T024815Z,FEATURE,baca zona silver,OK,8.88,1.19,37.6,2026-09-22T02:48:35.356247+00:00
1,run_20260922T024815Z,FEATURE,demonstrasi jendela geser,OK,1.58,0.36,37.2,2026-09-22T02:48:36.946893+00:00
2,run_20260922T024815Z,FEATURE,"bangun feature store untuk k=[4, 6, 8, 10]",OK,7.77,0.96,36.2,2026-09-22T02:48:44.737405+00:00
3,run_20260922T024815Z,FEATURE,ukur kejarangan vektor fitur,OK,56.25,5.58,30.6,2026-09-22T02:49:41.163885+00:00
4,run_20260922T024815Z,FEATURE,fitur untuk sekuens tanpa label,OK,35.41,0.01,30.6,2026-09-22T02:50:16.578150+00:00


Spark dihentikan. Proses Java tersisa: 9

Selesai. Lanjut ke 05_ml_segmen_subtipe.ipynb
